# Trajectory Sampling

This notebook provides an interactive single-atom trajectory test. The atom is placed on a spherical shell at a chosen radial distance and launched inward toward the origin. The cooling and repump beams from the geometry layer are reused directly, while the scattering and recoil dynamics come from the force layer.

In [1]:
%matplotlib widget

from pathlib import Path
import sys

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from pmot import (
    animation_samples,
    build_mot_beams,
    default_simulation_config,
    inward_radial_atom_state,
    project_paths,
    simulate_scattering_trajectory,
    trajectory_diagnostics,
)


In [2]:
PATHS = project_paths(PROJECT_ROOT)
CONFIG = default_simulation_config()
MOT_BEAMS = build_mot_beams(CONFIG)

DEFAULT_DURATION_MS = 3.0
DEFAULT_TIME_STEP_US = 2.0
DEFAULT_SEED = 11

pd.Series({
    'beam_count': len(MOT_BEAMS),
    'default_duration_ms': DEFAULT_DURATION_MS,
    'default_time_step_us': DEFAULT_TIME_STEP_US,
    'cooling_detuning_mhz': CONFIG.cooling.detuning_hz / 1e6,
    'repump_detuning_mhz': CONFIG.repump.detuning_hz / 1e6,
})


beam_count              12.0
default_duration_ms      3.0
default_time_step_us     2.0
cooling_detuning_mhz   -12.0
repump_detuning_mhz     -1.0
dtype: float64

## Interactive Inward Launch Study

The initial position is parameterized by spherical coordinates `(r, phi, theta)`, where:

- `r` is the initial radial distance from the origin
- `phi` is the azimuthal angle in the x-y plane, measured from +x toward +y
- `theta` is the polar angle measured from +z

The launch velocity is always directed inward along the negative radial direction, so the atom is shot toward the MOT center.

In [3]:
def draw_trajectory_study(
    radial_distance_mm: float = 15.0,
    initial_speed_m_per_s: float = 8.0,
    azimuth_deg: float = 0.0,
    polar_deg: float = 90.0,
    duration_ms: float = DEFAULT_DURATION_MS,
    time_step_us: float = DEFAULT_TIME_STEP_US,
    seed: int = DEFAULT_SEED,
):
    initial_state = inward_radial_atom_state(
        radial_distance_m=1e-3 * radial_distance_mm,
        speed_m_per_s=initial_speed_m_per_s,
        azimuth_rad=np.deg2rad(azimuth_deg),
        polar_rad=np.deg2rad(polar_deg),
    )
    trajectory = simulate_scattering_trajectory(
        MOT_BEAMS,
        initial_state,
        duration_s=1e-3 * duration_ms,
        time_step_s=1e-6 * time_step_us,
        seed=seed,
        active_transition='cooling',
    )
    diagnostics = trajectory_diagnostics(MOT_BEAMS, trajectory, active_transition='cooling')

    figure = plt.figure(figsize=(16, 11), constrained_layout=True)
    figure.patch.set_facecolor('#fbfaf6')
    grid = figure.add_gridspec(2, 2)
    velocity_axis = figure.add_subplot(grid[0, 0])
    scattering_axis = figure.add_subplot(grid[0, 1])
    position_axis = figure.add_subplot(grid[1, 0])
    trajectory_axis = figure.add_subplot(grid[1, 1], projection='3d')

    times_ms = [1e3 * value for value in diagnostics.times_s]

    for axis in (velocity_axis, scattering_axis, position_axis):
        axis.set_facecolor('#fbfaf6')
        axis.grid(True, alpha=0.28)

    velocity_axis.plot(times_ms, diagnostics.vx_m_per_s, color='#b91c1c', linewidth=1.9, label=r'$v_x$')
    velocity_axis.plot(times_ms, diagnostics.vy_m_per_s, color='#1d4ed8', linewidth=1.9, label=r'$v_y$')
    velocity_axis.plot(times_ms, diagnostics.vz_m_per_s, color='#15803d', linewidth=1.9, label=r'$v_z$')
    velocity_axis.plot(times_ms, diagnostics.speed_m_per_s, color='#111827', linewidth=1.4, linestyle='--', label='speed')
    velocity_axis.set_title('Velocity Components And Speed')
    velocity_axis.set_xlabel('Time [ms]')
    velocity_axis.set_ylabel('Velocity [m/s]')
    velocity_axis.legend(loc='best', frameon=True)

    scattering_axis.plot(times_ms, diagnostics.scattering_x_axis_per_s, color='#b91c1c', linewidth=1.9, label='x-axis beams')
    scattering_axis.plot(times_ms, diagnostics.scattering_y_axis_per_s, color='#1d4ed8', linewidth=1.9, label='y-axis beams')
    scattering_axis.plot(times_ms, diagnostics.scattering_z_axis_per_s, color='#15803d', linewidth=1.9, label='z-axis beams')
    scattering_axis.plot(times_ms, diagnostics.scattering_total_per_s, color='#111827', linewidth=1.4, linestyle='--', label='total')
    scattering_axis.set_title('Axis-Resolved Scattering Rates')
    scattering_axis.set_xlabel('Time [ms]')
    scattering_axis.set_ylabel(r'Scattering rate [s$^{-1}$]')
    scattering_axis.legend(loc='best', frameon=True)

    position_axis.plot(times_ms, diagnostics.x_mm, color='#b91c1c', linewidth=1.9, label='x')
    position_axis.plot(times_ms, diagnostics.y_mm, color='#1d4ed8', linewidth=1.9, label='y')
    position_axis.plot(times_ms, diagnostics.z_mm, color='#15803d', linewidth=1.9, label='z')
    position_axis.plot(times_ms, diagnostics.radius_mm, color='#7c3aed', linewidth=1.4, linestyle='--', label='radius')
    position_axis.set_title('Position Components And Radius')
    position_axis.set_xlabel('Time [ms]')
    position_axis.set_ylabel('Position [mm]')
    position_axis.legend(loc='best', frameon=True)

    trajectory_axis.set_facecolor('#fbfaf6')
    trajectory_axis.plot(diagnostics.x_mm, diagnostics.y_mm, diagnostics.z_mm, color='#0f766e', linewidth=2.0)
    trajectory_axis.scatter([diagnostics.x_mm[0]], [diagnostics.y_mm[0]], [diagnostics.z_mm[0]], color='#b91c1c', s=42, label='start')
    trajectory_axis.scatter([diagnostics.x_mm[-1]], [diagnostics.y_mm[-1]], [diagnostics.z_mm[-1]], color='#111827', s=42, label='end')
    trajectory_axis.set_title('3D Trajectory')
    trajectory_axis.set_xlabel('x [mm]')
    trajectory_axis.set_ylabel('y [mm]')
    trajectory_axis.set_zlabel('z [mm]')
    trajectory_axis.legend(loc='best', frameon=True)
    trajectory_axis.set_box_aspect((1.0, 1.0, 1.0))

    summary = pd.Series({
        'initial_position_mm': tuple(round(1e3 * value, 3) for value in initial_state.position_m),
        'initial_velocity_m_per_s': tuple(round(value, 3) for value in initial_state.velocity_m_per_s),
        'final_position_mm': tuple(round(1e3 * value, 3) for value in trajectory.positions_m[-1]),
        'final_velocity_m_per_s': tuple(round(value, 3) for value in trajectory.velocities_m_per_s[-1]),
        'peak_total_scattering_rate_per_s': round(max(diagnostics.scattering_total_per_s), 3),
        'final_speed_m_per_s': round(diagnostics.speed_m_per_s[-1], 6),
    })

    plt.show()
    display(summary)


controls = {
    'radial_distance_mm': widgets.FloatSlider(value=15.0, min=0.0, max=40.0, step=0.5, description='radius [mm]'),
    'initial_speed_m_per_s': widgets.FloatSlider(value=8.0, min=0.0, max=20.0, step=0.25, description='speed [m/s]'),
    'azimuth_deg': widgets.FloatSlider(value=0.0, min=0.0, max=360.0, step=1.0, description='azimuth [deg]'),
    'polar_deg': widgets.FloatSlider(value=90.0, min=0.0, max=180.0, step=1.0, description='polar [deg]'),
    'duration_ms': widgets.FloatSlider(value=DEFAULT_DURATION_MS, min=0.5, max=10.0, step=0.25, description='duration [ms]'),
    'time_step_us': widgets.FloatSlider(value=DEFAULT_TIME_STEP_US, min=0.5, max=10.0, step=0.5, description='dt [us]'),
    'seed': widgets.IntSlider(value=DEFAULT_SEED, min=0, max=200, step=1, description='seed'),
}

widgets.interact(draw_trajectory_study, **controls);


interactive(children=(FloatSlider(value=15.0, description='radius [mm]', max=40.0, step=0.5), FloatSlider(valu…

## Trajectory Animation

This widget reuses the same launch parameters and adds a play/slider control for the trajectory frame. The 3D point advances one timestep at a time and the black arrow shows the instantaneous velocity vector, so you can watch both position and speed evolve during cooling.

In [4]:
def build_trajectory_animation_widget(
    radial_distance_mm: float = 15.0,
    initial_speed_m_per_s: float = 8.0,
    azimuth_deg: float = 0.0,
    polar_deg: float = 90.0,
    duration_ms: float = DEFAULT_DURATION_MS,
    time_step_us: float = DEFAULT_TIME_STEP_US,
    seed: int = DEFAULT_SEED,
    max_animation_frames: int = 250,
):
    initial_state = inward_radial_atom_state(
        radial_distance_m=1e-3 * radial_distance_mm,
        speed_m_per_s=initial_speed_m_per_s,
        azimuth_rad=np.deg2rad(azimuth_deg),
        polar_rad=np.deg2rad(polar_deg),
    )
    trajectory = simulate_scattering_trajectory(
        MOT_BEAMS,
        initial_state,
        duration_s=1e-3 * duration_ms,
        time_step_s=1e-6 * time_step_us,
        seed=seed,
        active_transition='cooling',
    )
    diagnostics = trajectory_diagnostics(MOT_BEAMS, trajectory, active_transition='cooling')
    samples = animation_samples(
        trajectory,
        diagnostics,
        max_animation_frames=max_animation_frames,
    )

    x_mm = np.asarray(samples.x_mm, dtype=float)
    y_mm = np.asarray(samples.y_mm, dtype=float)
    z_mm = np.asarray(samples.z_mm, dtype=float)
    vx = np.asarray(samples.vx_m_per_s, dtype=float)
    vy = np.asarray(samples.vy_m_per_s, dtype=float)
    vz = np.asarray(samples.vz_m_per_s, dtype=float)
    speed = np.asarray(samples.speed_m_per_s, dtype=float)
    times_ms = np.asarray(samples.times_ms, dtype=float)
    frame_indices = np.asarray(samples.frame_indices, dtype=int)
    static_x = np.asarray(samples.static_x_mm, dtype=float)
    static_y = np.asarray(samples.static_y_mm, dtype=float)
    static_z = np.asarray(samples.static_z_mm, dtype=float)

    figure = plt.figure(figsize=(9.5, 8.5), constrained_layout=True)
    figure.patch.set_facecolor('#fbfaf6')
    axis = figure.add_subplot(111, projection='3d')
    axis.set_facecolor('#fbfaf6')
    axis.set_title('Animated 3D Trajectory And Velocity')
    axis.set_xlabel('x [mm]')
    axis.set_ylabel('y [mm]')
    axis.set_zlabel('z [mm]')
    axis.set_box_aspect((1.0, 1.0, 1.0))

    max_extent = max(1.0, float(np.max(np.abs(static_x))), float(np.max(np.abs(static_y))), float(np.max(np.abs(static_z))))
    plot_limit = 1.05 * max_extent
    axis.set_xlim(-plot_limit, plot_limit)
    axis.set_ylim(-plot_limit, plot_limit)
    axis.set_zlim(-plot_limit, plot_limit)

    axis.plot(static_x, static_y, static_z, color='#94a3b8', linewidth=1.2, alpha=0.70, label='full path')
    path_line, = axis.plot([x_mm[0]], [y_mm[0]], [z_mm[0]], color='#0f766e', linewidth=2.4, label='path to frame')
    atom_point, = axis.plot([x_mm[0]], [y_mm[0]], [z_mm[0]], marker='o', linestyle='None', color='#b91c1c', markersize=8, label='atom')
    axis.plot([x_mm[0]], [y_mm[0]], [z_mm[0]], marker='o', linestyle='None', color='#b91c1c', markersize=6, alpha=0.55, label='start')
    axis.plot([0.0], [0.0], [0.0], marker='x', linestyle='None', color='#111827', markersize=7, markeredgewidth=2, label='origin')

    velocity_scale = 0.35 * max_extent / max(1e-12, float(np.max(speed)))
    velocity_line, = axis.plot([x_mm[0], x_mm[0]], [y_mm[0], y_mm[0]], [z_mm[0], z_mm[0]], color='#111827', linewidth=2.2, label='velocity')
    velocity_tip, = axis.plot([x_mm[0]], [y_mm[0]], [z_mm[0]], marker='>', linestyle='None', color='#111827', markersize=5)
    info_label = widgets.HTML()

    def update_frame(frame_index: int):
        frame_index = int(np.clip(frame_index, 0, len(times_ms) - 1))
        x_now, y_now, z_now = x_mm[frame_index], y_mm[frame_index], z_mm[frame_index]
        path_line.set_data(x_mm[: frame_index + 1], y_mm[: frame_index + 1])
        path_line.set_3d_properties(z_mm[: frame_index + 1])
        atom_point.set_data([x_now], [y_now])
        atom_point.set_3d_properties([z_now])

        x_tip = x_now + velocity_scale * vx[frame_index]
        y_tip = y_now + velocity_scale * vy[frame_index]
        z_tip = z_now + velocity_scale * vz[frame_index]
        velocity_line.set_data([x_now, x_tip], [y_now, y_tip])
        velocity_line.set_3d_properties([z_now, z_tip])
        velocity_tip.set_data([x_tip], [y_tip])
        velocity_tip.set_3d_properties([z_tip])

        info_label.value = (
            f"<b>display frame:</b> {frame_index + 1}/{len(times_ms)} &nbsp; "
            f"<b>simulation index:</b> {frame_indices[frame_index]} &nbsp; "
            f"<b>time:</b> {times_ms[frame_index]:.3f} ms &nbsp; "
            f"<b>speed:</b> {speed[frame_index]:.4f} m/s"
            f"<br><b>position:</b> ({x_now:.3f}, {y_now:.3f}, {z_now:.3f}) mm"
        )
        figure.canvas.draw_idle()

    frame_slider = widgets.IntSlider(
        value=0,
        min=0,
        max=len(times_ms) - 1,
        step=1,
        description='frame',
        continuous_update=False,
        layout=widgets.Layout(width='650px'),
    )
    play = widgets.Play(
        value=0,
        min=0,
        max=len(times_ms) - 1,
        step=1,
        interval=20,
        description='play',
    )
    widgets.jslink((play, 'value'), (frame_slider, 'value'))
    frame_slider.observe(lambda change: update_frame(change['new']), names='value')
    axis.legend(loc='best', frameon=True)
    update_frame(0)
    display(widgets.VBox([widgets.HBox([play, frame_slider]), info_label]))
    plt.show()


animation_controls = {
    'radial_distance_mm': widgets.FloatSlider(value=15.0, min=0.0, max=40.0, step=0.5, description='radius [mm]'),
    'initial_speed_m_per_s': widgets.FloatSlider(value=8.0, min=0.0, max=20.0, step=0.25, description='speed [m/s]'),
    'azimuth_deg': widgets.FloatSlider(value=0.0, min=0.0, max=360.0, step=1.0, description='azimuth [deg]'),
    'polar_deg': widgets.FloatSlider(value=90.0, min=0.0, max=180.0, step=1.0, description='polar [deg]'),
    'duration_ms': widgets.FloatSlider(value=DEFAULT_DURATION_MS, min=0.5, max=10.0, step=0.25, description='duration [ms]'),
    'time_step_us': widgets.FloatSlider(value=DEFAULT_TIME_STEP_US, min=0.5, max=10.0, step=0.5, description='dt [us]'),
    'seed': widgets.IntSlider(value=DEFAULT_SEED, min=0, max=200, step=1, description='seed'),
    'max_animation_frames': widgets.IntSlider(value=250, min=50, max=600, step=25, description='max frames'),
}

widgets.interact(build_trajectory_animation_widget, **animation_controls)


interactive(children=(FloatSlider(value=15.0, description='radius [mm]', max=40.0, step=0.5), FloatSlider(valu…

<function __main__.build_trajectory_animation_widget(radial_distance_mm: float = 15.0, initial_speed_m_per_s: float = 8.0, azimuth_deg: float = 0.0, polar_deg: float = 90.0, duration_ms: float = 3.0, time_step_us: float = 2.0, seed: int = 11, max_animation_frames: int = 250)>